# US Sales Analysis with Salting Process

This notebook demonstrates the salting technique to handle data skew in PySpark.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, to_date, split, regexp_replace, round as spark_round
)

spark = SparkSession.builder \
    .appName('US Sales with Salting') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Load and Clean Data (Same as before)

In [ ]:
# Read and clean data (all steps combined)
raw_data_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/content/sample_data/US_Sales_Datasets.csv")

# Apply all cleaning transformations
cleaned_df = raw_data_df \
    .withColumn("Invoice Date", to_date(col("Invoice Date"), "dd-MM-yyyy")) \
    .withColumn("Gender", split(col("Product"), "'s ").getItem(0)) \
    .withColumn("Category", split(col("Product"), "'s ").getItem(1)) \
    .withColumn("Units Sold", regexp_replace(col("Units Sold"), ",", "").cast("Integer")) \
    .withColumn("Operating Margin", regexp_replace(col("Operating Margin"), "%", "").cast("Integer")) \
    .withColumn("Total Sales", (col("Units Sold") * col("Price per Unit")).cast("Double")) \
    .withColumn("Operating Profit", (col("Total Sales") * col("Operating Margin") / 100).cast("Double")) \
    .withColumnRenamed("Invoice Date", "Invoice_Date") \
    .withColumnRenamed("Total Sales", "Total_Sales") \
    .withColumnRenamed("Operating Profit", "Operating_Profit") \
    .withColumnRenamed("Units Sold", "Units_Sold")

cleaned_df.createOrReplaceTempView("sales")
print("Data loaded and cleaned successfully")
cleaned_df.show(5)

## Understanding Data Skew Problem

In [ ]:
# Check distribution of sales by city
city_distribution = cleaned_df.groupBy("City") \
    .count() \
    .orderBy(F.desc("count"))

print("Top 10 cities by transaction count:")
city_distribution.show(10)

# Check if there's heavy skew
total_records = cleaned_df.count()
top_city_count = city_distribution.first()[1]
skew_percentage = (top_city_count / total_records) * 100

print(f"\nTotal records: {total_records}")
print(f"Top city has {top_city_count} records ({skew_percentage:.2f}% of total)")
print(f"This indicates {'HIGH' if skew_percentage > 10 else 'LOW'} data skew")

## Traditional Aggregation (Without Salting)

In [ ]:
# Select only City and Total_Sales columns
city_df = cleaned_df.select(col("City"), col("Total_Sales"))

print("Sample city sales data:")
city_df.show(20)

In [ ]:
# Traditional aggregation for New York
import time

start_time = time.time()

ny_sales_traditional = city_df \
    .filter(col("City") == "New York") \
    .groupBy("City") \
    .agg(F.sum("Total_Sales").alias("Total_Sales"))

result_traditional = ny_sales_traditional.collect()
traditional_time = time.time() - start_time

print("Traditional Aggregation Result:")
ny_sales_traditional.show()
print(f"Time taken: {traditional_time:.4f} seconds")

## Salting Technique

### What is Salting?

Salting is a technique to handle data skew by:
1. Adding a random salt key to distribute skewed keys across multiple partitions
2. Performing aggregation with the salt key
3. Aggregating the salted results to get final result

This distributes the workload more evenly across executors.

In [ ]:
# Add salt column with random values (0-3)
random_val = F.round(F.rand() * 3)

salted_df = city_df.withColumn("salt", random_val)

print("Data with salt column:")
salted_df.show(20)

# Check salt distribution
print("\nSalt distribution:")
salted_df.groupBy("salt").count().orderBy("salt").show()

In [ ]:
# Register salted dataframe
salted_df.createOrReplaceTempView("SaltedTable")

### Step 1: Aggregate with salt key

In [ ]:
# First aggregation - group by City and salt
start_time_salted = time.time()

out1 = spark.sql("""
    SELECT 
        City, 
        salt, 
        SUM(Total_Sales) as Group_Total 
    FROM SaltedTable 
    WHERE City = 'New York' 
    GROUP BY City, salt
    ORDER BY salt
""")

print("Step 1: Aggregation by City and Salt:")
out1.show()

# This shows sales broken down by salt value
# Each salt group can be processed in parallel

### Step 2: Final aggregation (combine salt groups)

In [ ]:
# Final aggregation - sum across all salt values
out1.createOrReplaceTempView("SaltedResult")

final_result = spark.sql("""
    SELECT 
        City,
        SUM(Group_Total) as Final_Total_Sales
    FROM SaltedResult
    GROUP BY City
""")

result_salted = final_result.collect()
salted_time = time.time() - start_time_salted

print("Final Result after Salting:")
final_result.show()
print(f"Time taken with salting: {salted_time:.4f} seconds")

## Compare Results

In [ ]:
# Verify both methods give same result
print("Comparison of Results:")
print("=" * 50)
print(f"Traditional Method: ${result_traditional[0][1]:,.2f}")
print(f"Salting Method:     ${result_salted[0][1]:,.2f}")
print(f"\nDifference: ${abs(result_traditional[0][1] - result_salted[0][1]):,.2f}")
print("=" * 50)
print(f"\nTraditional Time: {traditional_time:.4f} seconds")
print(f"Salting Time:     {salted_time:.4f} seconds")

if salted_time < traditional_time:
    speedup = traditional_time / salted_time
    print(f"\nSalting is {speedup:.2f}x faster!")
else:
    print("\nNote: Salting benefits are more visible with larger datasets and higher skew")

## Salting for Multiple Cities

In [ ]:
# Apply salting for top 5 cities
top_cities = ['New York', 'Los Angeles', 'San Francisco', 'Miami', 'Chicago']

# Step 1: Aggregate with salt
multi_city_salted = salted_df \
    .filter(col("City").isin(top_cities)) \
    .groupBy("City", "salt") \
    .agg(F.sum("Total_Sales").alias("Group_Total"))

print("Step 1: Salted aggregation for multiple cities:")
multi_city_salted.orderBy("City", "salt").show(30)

# Step 2: Final aggregation
final_multi_city = multi_city_salted \
    .groupBy("City") \
    .agg(F.sum("Group_Total").alias("Total_Sales")) \
    .orderBy(F.desc("Total_Sales"))

print("\nFinal aggregated results:")
final_multi_city.show()

## When to Use Salting?

Use salting when:
- You have significant data skew (one or few keys dominate the dataset)
- Aggregation jobs are slow due to uneven partition distribution
- One executor is doing most of the work while others are idle

Salting works by:
1. **Distributing work**: Salt key splits skewed key across multiple partitions
2. **Parallel processing**: Each salt group can be processed independently
3. **Final merge**: Results are combined in a second aggregation step

**Trade-off**: Requires two aggregation stages instead of one, but can significantly improve performance for skewed data.

In [ ]:
# Stop Spark Session
spark.stop()